# 01 · 编程基础：五个动词

对应讲义第 04 部分 · 前置：00 环境检查

所有量子程序都由五个动作组成：**准备**（初始化量子态）、**演化**（作用量子门）、**组合**（搭建线路）、**测量**（采样得到比特串）、**估计**（计算期望值与梯度）。第 1–5 节逐一演示，第 6 节用它们写出第一个完整的量子算法。

**怎么用**：从上到下依次运行（`Shift + Enter`）。每个单元格都可以直接修改再运行，「试一试」指出了值得改的地方。练习单元格中的 `...` 需要换成你的代码，运行后 `check` 会立即告诉你对错。

In [ ]:
import numpy as np
from mindquantum import Circuit, H, X, RX, RY, RZ, UN, Measure
from mindquantum import Simulator, Hamiltonian, QubitOperator
from mqlab import check

## 1. 准备：模拟器保存量子态

`Simulator('mqvector', n)` 创建一个 n 比特的态矢量模拟器，初态为 $|0\cdots0\rangle$。

In [ ]:
sim = Simulator('mqvector', 1)   # 后端 + 比特数
print(sim)

sim.set_qs(np.array([1, 1]))     # 自动归一化为 (|0⟩ + |1⟩)/√2
print(sim.get_qs())              # 振幅向量
print(sim.get_qs(ket=True))      # 狄拉克记号

模拟器能直接“看见”量子态，真机做不到——这是调试量子程序最大的优势。试一试：把 `[1, 1]` 改成 `[1, -1]` 或 `[3, 4]`。

## 2. 演化：量子门就是幺正矩阵

In [ ]:
print(X.matrix())
print(H.matrix().round(3))
print(RX(np.pi / 2).matrix().round(3))

`.on(目标, 控制)` 把门作用到指定的比特上：**目标在前，控制在后**，与 Qiskit 的 `cx(控制, 目标)` 正好相反。

In [ ]:
sim = Simulator('mqvector', 2)
sim.apply_gate(H.on(0))          # H 作用在 q0
sim.apply_gate(X.on(1, 0))       # 目标 q1，控制 q0（CNOT）
print(sim.get_qs(ket=True))

试一试：把 `X.on(1, 0)` 写反成 `X.on(0, 1)`，还会得到 Bell 态吗？

<details><summary>答案</summary>

不会。控制比特变成了 q1，而 q1 一直是 0，CNOT 什么也不做，末态是 $(|00\rangle + |01\rangle)/\sqrt{2}$：q0 处于叠加，两个比特没有纠缠。代码不报错、结果却悄悄变了，这是最常见的一类错误。
</details>

## 3. 组合：用 Circuit 把门组成线路

线路与模拟器是分开的：同一个 `Circuit` 可以画图、求矩阵，也可以交给不同的后端运行。

In [ ]:
circ = Circuit()
circ += H.on(0)
circ += X.on(1, 0)
circ += RY(np.pi / 4).on(2, 1)   # 受控 RY：目标 q2，控制 q1
circ.svg()

In [ ]:
sim = Simulator('mqvector', circ.n_qubits)
sim.apply_circuit(circ)
print(sim.get_qs(ket=True))

### 先预测：比特串怎么读？

3 个比特，只对 q0 作用 X。模拟器会输出 `|100⟩` 还是 `|001⟩`？先写下答案，再运行。

In [ ]:
sim = Simulator('mqvector', 3)
sim.apply_gate(X.on(0))          # 只翻转 q0
print(sim.get_qs(ket=True))

MindSpore Quantum 使用**小端序**：比特串从右往左读，最右边是 q0，与二进制数“低位在右”一致。Qiskit 同样是小端序；很多教材和 PennyLane 把 q0 写在最左。比特序是跨框架、跨教材最常见的错误来源。

### 练习 1 · 四个 Bell 态

补全 `bell(kind)`，返回制备对应 Bell 态的两比特线路：

| kind | 量子态 |
|---|---|
| `'phi+'` | (\|00⟩ + \|11⟩)/√2 |
| `'phi-'` | (\|00⟩ − \|11⟩)/√2 |
| `'psi+'` | (\|01⟩ + \|10⟩)/√2 |
| `'psi-'` | (\|01⟩ − \|10⟩)/√2 |

提示：从 $\Phi^+$ 的线路（H + CNOT）出发，想一想在 H 之前加什么门，能改变两个比特的关系或相对相位。

In [ ]:
def bell(kind):
    circ = Circuit()
    # 练习 1：根据 kind 搭建线路
    ...
    return circ

check.bell_states(bell)

## 4. 测量：多次采样，得到比特串的分布

`measure_all()` 给每个比特加上测量门；`sampling` 把线路运行 `shots` 次并统计结果；`res.svg()` 画出分布。`seed` 固定随机数，便于复现。

In [ ]:
circ = Circuit([H.on(0), X.on(1, 0)])
circ.measure_all()               # 每个比特加测量门

sim = Simulator('mqvector', 2)
res = sim.sampling(circ, shots=1000, seed=2026)
res.svg()

试一试：

- 把 `shots=1000` 改成 `10`，再改成 `100000`：统计误差按 $1/\sqrt{N}$ 缩小，精度提高 10 倍，要多采样 100 倍；
- 删掉 `seed=2026`，多运行几次：每次的计数都不一样。

注意：`sampling` 从模拟器的**当前态**出发。之前用 `apply_circuit` 改变过模拟器的话，先 `sim.reset()`。

## 5. 估计：期望值、参数与梯度

期望值 $\langle\psi|H|\psi\rangle$ 是测量结果的“平均”，也是变分算法的燃料。模拟器可以精确计算；真机上要靠大量采样来估计。

In [ ]:
bell = Circuit([H.on(0), X.on(1, 0)])
sim = Simulator('mqvector', 2)
for term in ['Z0', 'Z1', 'Z0 Z1']:
    ham = Hamiltonian(QubitOperator(term))
    e = sim.get_expectation(ham, bell)
    print(f'<{term}> = {e.real:+.3f}')

单看每个比特，0、1 各一半，期望为 0；两个比特的乘积却总是 +1——完全相关。

试一试：先预测 $\langle X_0X_1\rangle$ 与 $\langle Y_0Y_1\rangle$，再把列表改成 `['X0 X1', 'Y0 Y1']` 运行。

<details><summary>答案</summary>

+1 与 −1。$\Phi^+$ 在 X 方向上同样完全相关，在 Y 方向上完全反相关：纠缠不只是“同为 0 或同为 1”，在任何方向上测量都有关联。
</details>

### 参数化线路：参数先用名字占位，运行时再赋值

In [ ]:
circ = Circuit([RX('theta').on(0)])   # 参数用名字占位
print(circ.params_name)

ham = Hamiltonian(QubitOperator('Z0'))
sim = Simulator('mqvector', 1)
for t in [0, np.pi / 2, np.pi]:
    e = sim.get_expectation(ham, circ, pr={'theta': t})
    print(f'theta = {t:.3f}   <Z> = {e.real:+.3f}')

### 梯度：一次给出期望值与导数

`get_expectation_with_grad` 返回一个函数：输入参数值，输出期望值和对每个参数的导数。它是变分算法（实验 C、D）的训练引擎。

In [ ]:
grad_ops = sim.get_expectation_with_grad(ham, circ)
f, g = grad_ops(np.array([np.pi / 3]))     # θ = π/3
print(f'f = {f.real.item():.3f}')          # cos(π/3)
print(f'df/dθ = {g.real.item():.3f}')      # −sin(π/3)

## 6. 第一个完整算法：Bernstein–Vazirani

**问题**：黑盒函数 $f(x) = s\cdot x \bmod 2$ 藏着一个 n 位秘密串 s。经典算法依次输入 0…01、0…10、……，每次只能问出 s 的 1 位，需要 n 次；量子算法只需 **1 次**。

**直觉**：关键不是“同时算出所有 f(x)”，而是把 f(x) 写进相位，再让相位干涉出 s。把结果写进相位的机制叫相位反冲，先用两个比特看清它。

### 先预测：CNOT 会改变控制比特吗？

q0 处于 $|+\rangle$、q1 处于 $|-\rangle$，作用 CNOT（q0 控制）之后，再对 q0 作用 H。测量 q0，结果是 A 总是 0、B 总是 1，还是 C 各占一半？

In [ ]:
circ = Circuit([H.on(0), X.on(1), H.on(1)])  # q0: |+⟩, q1: |−⟩
circ += X.on(1, 0)                           # CNOT，q0 控制
circ += H.on(0)
sim = Simulator('mqvector', 2)
sim.apply_circuit(circ)
print(sim.get_qs(ket=True))

<details><summary>答案</summary>

B：q0 总是 1（两项的最右一位都是 1），q1 仍是 $|-\rangle$。因为 $X|-\rangle = -|-\rangle$，CNOT 作用在 $|x\rangle|-\rangle$ 上只留下一个负号 $(-1)^x$，这个负号落在控制比特上：q0 从 $|+\rangle$ 变成 $|-\rangle$，再经 H 变成 $|1\rangle$。这就是**相位反冲**：“控制”与“目标”的说法只在计算基下成立。
</details>

### BV 线路

数据比特 $q_0 \ldots q_{n-1}$，辅助比特 $q_n$：

1. 辅助比特先用 X 翻到 $|1\rangle$，再和所有数据比特一起作用 H，辅助比特变成 $|-\rangle$；
2. Oracle：s 的第 i 位为 1，就从 $q_i$ 连一个 CNOT 到辅助比特。每个 CNOT 给 $|x\rangle$ 反冲一个 $(-1)^{x_i}$，合起来正是 $(-1)^{s\cdot x}$；
3. 数据比特再作用一次 H，把相位干涉成 $|s\rangle$；
4. 测量数据比特。

下面是完整的程序（与讲义相同）：只采样 1 次，再画出线路。

In [ ]:
s = '1011'                     # 秘密串，按 q3 q2 q1 q0 书写
n = len(s)
circ = Circuit()
circ += X.on(n)                # 辅助比特 q4 → |1⟩
circ += UN(H, n + 1)           # 全部叠加，q4 → |−⟩
for i, b in enumerate(reversed(s)):
    if b == '1':
        circ += X.on(n, i)     # Oracle：s 的第 i 位为 1
circ += UN(H, n)
circ += UN(Measure(), n)       # 只测量 q0…q3

sim = Simulator('mqvector', n + 1)
print(sim.sampling(circ, shots=1).data)
circ.svg()

### 现场改一改：先预测，再运行

直接修改上面的单元格，每次改一处，先写下预测再运行：

1. **换一个秘密串**：`s = '0110'`。Oracle 会从哪些比特连 CNOT？　A q0、q3　B q1、q2　C q2、q3
2. **秘密串加长到 8 位**：`s = '10110010'`。要采样几次才能读出 s？　A 1 次　B 8 次　C 256 次
3. **忘了把辅助比特翻成 $|1\rangle$**：删掉 `circ += X.on(n)` 这一行。输出会是什么？　A 仍是 `1011`　B `0000`　C 随机的比特串

想确认结果是不是确定的，把 `shots=1` 改成 `shots=100`。

<details><summary>答案</summary>

1. B：`'0110'` 按 q3 q2 q1 q0 书写，为 1 的是 q1、q2，线路图里的两个 CNOT 正是从它们出发。
2. A：仍然只要 1 次。线路随 n 线性变长，查询次数始终是 1；经典算法需要 8 次。
3. B：`0000`，而且每次都是。辅助比特停在 $|+\rangle$，它是 X 的本征态、本征值为 +1，CNOT 不留下任何相位，两层 H 相互抵消。辅助比特必须处于 $|-\rangle$，相位反冲才会发生。
</details>

## 常见错误清单

| 现象 | 原因 | 正确做法 |
|---|---|---|
| CNOT 方向反了 | 按 Qiskit 习惯写成 `X.on(控制, 目标)` | `X.on(目标, 控制)` |
| 比特串读反了 | 以为 q0 在最左 | 小端序：q0 在最右 |
| 第二次采样结果不对 | 模拟器保留了上一次的末态 | `sim.reset()` 后再运行 |
| 报错缺少参数 | `pr` 字典漏了某个参数名 | 用 `circ.params_name` 核对 |
| 一次采样就下结论 | 结果本身是随机的 | 先用 `get_qs()` 看理想态，再采样 |

## 课后挑战 · IQP 编码线路

按下图搭建 4 比特线路，参数名依次为 `alpha0` … `alpha6`：

![IQP 编码线路](images/iqp.svg)

然后：

1. 随机给 7 个参数赋值，打印线路运行后的量子态；
2. 采样 10,000 次，观察分布；
3. 思考：为什么量子态看起来毫无规律，采样分布却几乎均匀？如果在线路末尾给每个比特再加一个 H，分布还均匀吗？

这条线路正是实验 D 中把数据写进量子态的“编码线路”。

In [ ]:
circ = Circuit()
# 挑战：按图搭建线路
...
check.iqp(circ)

In [ ]:
if len(circ) == 0:
    print('请先完成挑战')
else:
    circ = circ.remove_measure()
    rng = np.random.default_rng(0)
    pr = dict(zip(circ.params_name, rng.uniform(0, 2 * np.pi, len(circ.params_name))))
    sim = Simulator('mqvector', 4)
    sim.apply_circuit(circ, pr=pr)
    print(sim.get_qs(ket=True))
    sim.reset()
    res = sim.sampling(circ + UN(Measure(), 4), pr=pr, shots=10000, seed=1)
    display(res.svg())